# Song selector for prompt design

Browse songs from the dataset one at a time, filtered by label (`M` / `NM`) and country of origin (Spain / Latin America). Use **Previous** / **Next** to step through the filtered list and read the full lyrics of each song. The goal is to make it easy to pick a good candidate song for a future prompt.

In [1]:
import pandas as pd

train_df = pd.read_csv("../data/train.csv").assign(split="train")
test_df = pd.read_csv("../data/test.csv").assign(split="test")
df = pd.concat([train_df, test_df], ignore_index=True)

LANGUAGE_LABELS = {"es_ES": "Spain", "es_LATAM": "Latin America"}
df["country"] = df["language"].map(LANGUAGE_LABELS)

df.head(3)

,song_id,song_title,artist_id,artist_name,lyrics,language,is_misogynistic,type_sexualization,type_violence,type_hate,has_gender_stereotype,split,country
0,4175,Rompe,294,Daddy Yankee,You know\nLos capos están ready\nLas mamis est...,es_LATAM,M,1.0,0.0,0.0,Y,train,Latin America
1,87563,Me Estás Tentando,381,Wisin & Yandel,La Mente Maestra\nHace rato que tu mirada me a...,es_LATAM,M,1.0,0.0,0.0,Y,train,Latin America
2,90914,Obsesión,548,Aventura (Ft. Judy Santos),"Something flavored, Aventura\nHello?\nShh, sol...",es_LATAM,M,1.0,0.0,0.0,Y,train,Latin America


## Interactive browser

- Check/uncheck **M** / **NM** to include or exclude that label.
- Check/uncheck **Spain** / **Latin America** to include or exclude that country.
- Use **◀ Previous** / **Next ▶** to move through the filtered songs.

In [2]:
import ipywidgets as widgets
from IPython.display import display

label_m = widgets.Checkbox(value=True, description="M (misogynistic)")
label_nm = widgets.Checkbox(value=True, description="NM (not misogynistic)")
country_spain = widgets.Checkbox(value=True, description="Spain")
country_latam = widgets.Checkbox(value=True, description="Latin America")

prev_button = widgets.Button(description="\u25c0 Previous", layout=widgets.Layout(width="120px"))
next_button = widgets.Button(description="Next \u25b6", layout=widgets.Layout(width="120px"))
counter_label = widgets.Label()
output = widgets.Output()

current_index = {"value": 0}


def get_filtered_songs():
    allowed_labels = [l for l, checked in [("M", label_m.value), ("NM", label_nm.value)] if checked]
    allowed_countries = [c for c, checked in [("Spain", country_spain.value), ("Latin America", country_latam.value)] if checked]
    subset = df[df["is_misogynistic"].isin(allowed_labels) & df["country"].isin(allowed_countries)]
    return subset.sort_values("song_title").reset_index(drop=True)


def show_song(*_):
    subset = get_filtered_songs()
    n = len(subset)
    prev_button.disabled = n == 0 or current_index["value"] <= 0
    next_button.disabled = n == 0 or current_index["value"] >= n - 1

    output.clear_output(wait=True)
    with output:
        if n == 0:
            counter_label.value = "0 songs match the current filters"
            print("No songs found for this combination of filters.")
            return

        counter_label.value = f"Song {current_index['value'] + 1} of {n}"
        row = subset.iloc[current_index["value"]]
        print(f"song_id={row['song_id']}  label={row['is_misogynistic']}  country={row['country']}  split={row['split']}")
        print()
        print(f"{row['song_title']} \u2014 {row['artist_name']}")
        print("=" * 60)
        print(row["lyrics"])


def reset_and_show(*_):
    current_index["value"] = 0
    show_song()


def go_previous(_):
    if current_index["value"] > 0:
        current_index["value"] -= 1
        show_song()


def go_next(_):
    subset = get_filtered_songs()
    if current_index["value"] < len(subset) - 1:
        current_index["value"] += 1
        show_song()


for checkbox in (label_m, label_nm, country_spain, country_latam):
    checkbox.observe(reset_and_show, names="value")

prev_button.on_click(go_previous)
next_button.on_click(go_next)

reset_and_show()

display(widgets.VBox([
    widgets.HTML("<b>Label</b>"),
    widgets.HBox([label_m, label_nm]),
    widgets.HTML("<b>Country of origin</b>"),
    widgets.HBox([country_spain, country_latam]),
    widgets.HBox([prev_button, next_button, counter_label]),
    output,
]))